In [1]:
import time
from pprint import pprint
from tqdm import tqdm

#avenieca sdk imports
from avenieca import Signal
from avenieca.producers import Event
from avenieca.api.model import *
from avenieca.api.eca import ECA

#local file importse
from config import *

def prettyprint(res, status):
    try:
        pprint(res.__dict__)
    except:
        print(res)
    print(status)

load_dotenv()
data_path = os.getenv("DATA_PATH")
url = '%s/bitcoin.csv' % data_path

In [2]:
import pandas as pd
data = pd.read_csv(url)

In [3]:
btc_data = data["price"].values
len(btc_data)

3091

### Stream to ECA

In [5]:
btc_broker_config = btc_twin_config.broker_config
btc_event = Event(config=btc_broker_config)

btc_data = data["price"].values
btc_data_train = btc_data[0:3082]
btc_data_test = btc_data[3082:]
print(btc_data_train[-1])

26469.581684007026


In [ ]:
for i in tqdm(range(0, len(btc_data))):
    btc_signal = Signal(
        state=[float(btc_data[i])]
    )
    
    future = btc_event.publish(btc_signal)
    _ = future.get(timeout=60)
    
    time.sleep(0.2)

### Next state predictions for an ESS

In [21]:
eca_server = os.getenv("ECA_SERVER")

config = Config(uri=eca_server, username=username, password=password)
eca = ECA(config)


In [ ]:
res, status = eca.ess.search(data=Search(
    module_id="temperature",
    state=[25.0],
    limit=1
))
prettyprint(res, status)

In [ ]:
nsr = NextStateRequest(
    module_id="temperature",
    recall=3,
    range=2,
    n=3,
    status='n',
    current_state=5,
    previous_state=[4, 7]
)
res, status = eca.cortex.predictions_raw(data=nsr)
prettyprint(res, status)

### Next state predictions for an Aggregate

In [8]:
res, status = eca.ess.search(data=Search(
    module_id="occupancy",
    state=[10.0],
    limit=1
))
prettyprint(res, status)

[SearchResult(score=0.0, ess=ESSResponse(id=7, state=[10.0], module_id='occupancy', valence=-90.0, created_at='2023-06-11T14:56:08.977245Z', updated_at='2023-06-11T14:56:32.953427Z', avg_ess_valence=0.0, total_ess_score=0, avg_ess_score=0, score=3, embedding_input=None, aggregate_id=[], aggregate_valence=[], aggregate_score=[], aggregate_module_id=[], aggregate_shape=[], aggregate_context=[], aggregate_emb_inp=[], context=None))]
200


In [ ]:
ess_air_conditioner = ESSResponse(
    id=5,
    state=[25.0],
    module_id='air_conditioner',
    valence=90.0,
    score=9,
)
ess_occupancy = ESSResponse(
    id=7,
    state=[10.0],
    module_id='occupancy',
    valence=-90.0,
    score=3,
)
ess_purifier = ESSResponse(
    id=3,
    state=[2.0],
    module_id='purifier',
    valence=90.0,
    score=14,
)

In [ ]:
res, status = eca.ess.search(data=Search(
    module_id="temperature",
    state=[29.0],
    limit=1
))
prettyprint(res, status)

In [9]:
ess_in = ESSInsert(
    module_id="air_quality_index",
    state=[90.0],
    valence=-90,
    score=1,
    context=None,
    embedding_input=None
)
ess_air_quality_index, status = eca.ess.create(data=ess_in)
prettyprint(ess_air_quality_index, status)

ess_in = ESSInsert(
    module_id="temperature",
    state=[29.0],
    valence=-90,
    score=1,
    context=None,
    embedding_input=None
)
ess_temperature, status = eca.ess.create(data=ess_in)
prettyprint(ess_temperature, status)

{'aggregate_context': [],
 'aggregate_emb_inp': [],
 'aggregate_id': [],
 'aggregate_module_id': [],
 'aggregate_score': [],
 'aggregate_shape': [],
 'aggregate_valence': [],
 'avg_ess_score': 0,
 'avg_ess_valence': 0.0,
 'context': None,
 'created_at': '2023-06-11T14:58:19.127550Z',
 'embedding_input': None,
 'id': 18,
 'module_id': 'air_quality_index',
 'score': 1,
 'state': [90.0],
 'total_ess_score': 0,
 'updated_at': '2023-06-11T14:58:19.127550Z',
 'valence': -90.0}
201
{'aggregate_context': [],
 'aggregate_emb_inp': [],
 'aggregate_id': [],
 'aggregate_module_id': [],
 'aggregate_score': [],
 'aggregate_shape': [],
 'aggregate_valence': [],
 'avg_ess_score': 0,
 'avg_ess_valence': 0.0,
 'context': None,
 'created_at': '2023-06-11T14:58:19.265674Z',
 'embedding_input': None,
 'id': 13,
 'module_id': 'temperature',
 'score': 1,
 'state': [29.0],
 'total_ess_score': 0,
 'updated_at': '2023-06-11T14:58:19.265674Z',
 'valence': -90.0}
201


In [ ]:
aggregate_insert = ESSInsert(
    module_id="aggregate001",
    state=[],
    valence=0.0, 
)
agg_in = create_aggregate_from_ess(
    [
        ess_air_conditioner,
        ess_air_quality_index,
        ess_occupancy,
        ess_purifier,
        ess_temperature
    ],
    aggregate_insert)
res, status = eca.ess.create(data=agg_in)
prettyprint(res, status)

In [12]:
aggregate_insert = ESSInsert(
    module_id="aggregate001",
    state=[],
    valence=0.0, 
    aggregate_id=[5, 18, 7, 3, 13],
    aggregate_module_id=["air_conditioner", "air_quality_index", "occupancy", "purifier", "temperature"]
)
res, status = eca.ess.create(data=aggregate_insert)
prettyprint(res, status)

{'aggregate_context': [None, None, None, None, None],
 'aggregate_emb_inp': [None, None, None, None, None],
 'aggregate_id': [5, 18, 7, 3, 13],
 'aggregate_module_id': ['air_conditioner',
                         'air_quality_index',
                         'occupancy',
                         'purifier',
                         'temperature'],
 'aggregate_score': [9, 1, 3, 14, 1],
 'aggregate_shape': [1, 1, 1, 1, 1],
 'aggregate_valence': [90.0, -90.0, -90.0, 90.0, -90.0],
 'avg_ess_score': 5,
 'avg_ess_valence': -18.0,
 'context': None,
 'created_at': '2023-06-11T15:06:25.726206Z',
 'embedding_input': None,
 'id': 33,
 'module_id': 'aggregate001',
 'score': 0,
 'state': [25.0, 90.0, 10.0, 2.0, 29.0],
 'total_ess_score': 28,
 'updated_at': '2023-06-11T15:06:25.726206Z',
 'valence': -90.0}
201


In [27]:
res, status = eca.ess.get_one(module_id="aggregate001",db_id=21)
prettyprint(res, status)

{'aggregate_context': [None, None, None, None, None],
 'aggregate_emb_inp': [None, None, None, None, None],
 'aggregate_id': [3, 14, 12, 2, 7],
 'aggregate_module_id': ['air_conditioner',
                         'air_quality_index',
                         'occupancy',
                         'purifier',
                         'temperature'],
 'aggregate_score': [9, 1, 4, 18, 3],
 'aggregate_shape': [1, 1, 1, 1, 1],
 'aggregate_valence': [90.0, 90.0, -90.0, 90.0, -90.0],
 'avg_ess_score': 7,
 'avg_ess_valence': 18.0,
 'context': None,
 'created_at': '2023-06-11T23:25:53.279098Z',
 'embedding_input': None,
 'id': 21,
 'module_id': 'aggregate001',
 'score': 1,
 'state': [18.0, 44.0, 12.0, 1.0, 22.0],
 'total_ess_score': 35,
 'updated_at': '2023-06-11T23:25:53.370920Z',
 'valence': 90.0}
200


In [22]:
res, status = eca.ess.search(data=Search(
    module_id="aggregate001",
    state=[20.0, 70.0, 10.0, 2.0, 29.0],
    limit=1
))
prettyprint(res, status)

[SearchResult(score=6.78233, ess=ESSResponse(id=4, state=[18.0, 70.0, 15.0, 1.0, 25.0], module_id='aggregate001', valence=-90.0, created_at='2023-06-11T23:25:34.781855Z', updated_at='2023-06-11T23:25:35.141833Z', avg_ess_valence=-18.0, total_ess_score=33, avg_ess_score=6, score=1, embedding_input=None, aggregate_id=[3, 6, 6, 2, 5], aggregate_valence=[90.0, -90.0, -90.0, 90.0, -90.0], aggregate_score=[9, 1, 3, 18, 2], aggregate_module_id=['air_conditioner', 'air_quality_index', 'occupancy', 'purifier', 'temperature'], aggregate_shape=[1, 1, 1, 1, 1], aggregate_context=[None, None, None, None, None], aggregate_emb_inp=[None, None, None, None, None], context=None))]
200


In [ ]:
nsr = NextStateRequest(
    module_id="aggregate001",
    recall=3,
    range=2,
    n=3,
    status='n',
    current_state=33,
    previous_state=[20, 25]
)
res, status = eca.cortex.predictions_raw(data=nsr)
prettyprint(res, status)

### Natural Language Retrieval

In [ ]:
retrieval = RetrievalRequest(
    query="what is the temperature and the air quality index on the 31st of may at around 1:50pm?"
)
res, status = eca.retrieval.query(data=retrieval)
prettyprint(res, status)